# Processing MITGCM data for the Spatial Scale Analysis 

**Purpose**: Code for producing data for Spatial decorrelation scale analysis. Here, we will generate spatial maps of temperature, salinity, and velocity at a single depth off the coast of Point Conception. 

**Luke Colosi | lcolosi@ucsd.edu**

Import python libraries

In [1]:
import sys
import numpy as np
import xarray as xr
from xmitgcm import open_mdsdataset

Set data analysis parameters

In [3]:
# Model parameters 
delta_t = 150  # Time steps of the raw model run (by raw, I mean the time increments that the model is ran at, not time increments that the diagnostics are output at). Units: seconds

# Set time and space parameters  
depth = 0                                                      # Specifies the depth level for the spatial map (units: meters). Options: 0, 10, 75, 1000
lat_bnds  = [33.0, 35.0]                                          # Specifies the latitude bounds for the region to analyze
lon_bnds  = [237.0, 240.0]                                        # Specifies the longitude bounds for the region to analyze
encoding  = {'time': {'units': 'seconds since 2015-12-01 2:00'}}  # Specifies the start time of the model run

# Set path to project directory
PATH_GRID   = '/data/SO2/SWOT/GRID/BIN/'                    # Space and time grid of the model 
PATH_OUTPUT = '/data/SO2/SWOT/MARA/RUN4_LY/DIAGS_HRLY/'     # Diagnostics of the model
PATH_nc     = '/data/SO3/lcolosi/mitgcm/SWOT_MARA_RUN4_LY/' # Directory to save netCDFs 
file_dim    = '2D'                                          # Set the dimension of the data (to include the depth or not where 3D is T,S,drhodr,vel and 2D is etan)

Load the grid and diagnostics data into a python structure. The diagnostics that we will be looking at include: 

**3D Fields** 
1. **Potential Temperature** $\theta$: $^\circ C$
2. **Salinity** $S$: $g/kg$
3. **Stratification** $\frac{d\sigma}{dz}$: $kg/m^4$
4. **Zonal, meridional, and vertical velocity components**  $\textbf{u} = (u,v,w)$: $m/s$  

**2D Fields** 
1. **Sea Surface Height Anomaly** $\eta$ (ETAN): cm

In [ ]:
# Create dataset 
ds = open_mdsdataset(
    PATH_OUTPUT,                    # File path where the model output data is stored (.data and .meta files)
    PATH_GRID,                      # File path to the grid data of the model 
    iters='all',                    # Specifies which iteration of the data to load
    delta_t=delta_t, 
    ignore_unknown_vars=False,      # Specifies whether to ignore any unknown variables that may appear in the dataset
    prefix=['diags_' + file_dim],   # List of prefixes to filter the variables in the dataset
    ref_date="2015-01-01 02:00:00", # Specifies the starting point of the simulation time (which may include the spin up time before diagnostics are output)
    geometry='sphericalpolar'       # Specifies the  grid's geometry is spherical-polar. 
)

# Convert all variables and coordinates in the dataset to little-endian (the format how multi-byte values are stored into memory)

#--- Variables ---#
for var in ds.data_vars:
    if ds[var].dtype.byteorder == '>' or (ds[var].dtype.byteorder == '=' and sys.byteorder == "big"):  # Check if big-endian
        ds[var] = ds[var].astype(ds[var].dtype.newbyteorder('<'))

#--- Coordinates ---# 
for coord in ds.coords:
    if ds[coord].dtype.byteorder == '>'or (ds[coord].dtype.byteorder == '=' and sys.byteorder == "big"):  # Check if big-endian
        ds[coord] = ds[coord].astype(ds[coord].dtype.newbyteorder('<'))

Slice array based on longitude and latitude bounds of the region

In [11]:
if file_dim == '3D':
    print(f"Extracting 3D fields...")

    # Obtain the depth coordinate 
    depth_levels = abs(ds['Z'].values)

    # Find the index of the depth level closest to depth
    depth_idx = np.argmin(np.abs(depth_levels - depth))

    # Check the actual depth value you're selecting
    actual_depth = depth_levels[depth_idx]
    print(f"Selected depth: {actual_depth} m at index {depth_idx}")

    # Extract scalar fields 
    theta = ds['THETA'].isel(Z=depth_idx).sel(YC=slice(*lat_bnds), 
                                            XC=slice(*lon_bnds))
    salt  = ds['SALT'].isel(Z=depth_idx).sel(YC=slice(*lat_bnds), 
                                            XC=slice(*lon_bnds))
    uvel  = ds['UVEL'].isel(Z=depth_idx).sel(YC=slice(*lat_bnds), 
                                            XG=slice(*lon_bnds))
    vvel  = ds['VVEL'].isel(Z=depth_idx).sel(YG=slice(*lat_bnds), 
                                            XC=slice(*lon_bnds))

elif file_dim == '2D':
    print(f"Extracting 2D fields...")

    # Extract scalar fields 
    etan = ds['ETAN'].sel(YC=slice(*lat_bnds), 
                          XC=slice(*lon_bnds))

Extracting 2D fields...


Save data in netcdf files

In [13]:
# Set the dictionary of variables to save
if file_dim == '3D':
    vars_to_save = {
        'THETA': theta,
        'SALT': salt,
        'UVEL': uvel,
        'VVEL': vvel
    }
elif file_dim == '2D':
    vars_to_save = {
        'ETAN': etan
    }

# Loop through each variable and save efficiently
for var_name, da in vars_to_save.items():
    
    # Chunk along time for faster write (adjust chunk size if needed)
    if 'time' in da.dims:
        da = da.chunk({'time': 1000})
    
    # Load into memory before saving (triggers computation)
    da = da.load()
    
    # Print status
    print(f"Saving {var_name}...")

    # Save to NetCDF file
    if file_dim == '3D':
        da.to_netcdf(
            f"{PATH_nc}{var_name}_CCS4_hrly_map_depth_{abs(int(actual_depth))}m.nc",
            engine='scipy',                # Fastest NetCDF writer (writes NetCDF3)
            format='NETCDF3_64BIT',        # Simple + compatible format
            encoding=encoding            
        )
    elif file_dim == '2D':
        da.to_netcdf(
            f"{PATH_nc}{var_name}_CCS4_hrly_map_2D.nc",
            engine='scipy',                # Fastest NetCDF writer (writes NetCDF3)
            format='NETCDF3_64BIT',        # Simple + compatible format
            encoding=encoding            
        )


Saving ETAN...
